# Advanced 04 — Hybrid Production Architecture

Build an application-owned architecture control plane for Northstar Commerce. The governing rule is: **use the smallest appropriate amount of model-driven autonomy**. Classifiers and frameworks propose or execute; trusted application policy owns authority, budgets, transitions, validation, and completion.

This notebook is deterministic, credential-free, and uses the same `policy.py` and `lab.py` exercised by the focused test suite.

## 1. Setup and learning contract

We will separate classification, architecture admission, authorization, execution, and result validation. Keeping those boundaries distinct is the core production lesson.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

COURSE_DIR = Path.cwd() / "curriculum/advanced/04-hybrid-production-architecture"
if not COURSE_DIR.exists():
    COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR.resolve()))

from policy import *
from lab import *
from framework_adapters import adapter_status

pd.set_option("display.max_colwidth", 80)
print("Course modules loaded; no API key or network required.")

## 2. The six Northstar requests

The same fixture lets us compare architectures rather than comparing unrelated demos. A valid architecture may still be more costly or complex than the best compliant architecture.

In [ ]:
pd.DataFrame([{"scenario": key, "request": value} for key, value in REQUESTS.items() if key != "unknown_destructive"])

## 3. Trusted request context

Identity, tenant, roles, capabilities, data classification, policy version, and deadline arrive from trusted application context. Models cannot add or mutate them. Pydantic rejects extra fields and freezes the model.

In [ ]:
context = build_context("notebook-request-001")
context.model_dump(mode="json")

## 4. Classification is a proposal

The offline rule fixture could be replaced by ML or an LLM, but the result remains a proposal. Application policy applies recognized high-risk overrides, rejects data downgrades, and treats destructive unknowns as `HIGH_RISK_UNKNOWN` with no privileged route.

In [ ]:
classification_rows = []
for name, text in REQUESTS.items():
    item_context = build_context(f"classify-{name}")
    proposed = propose_classification(text, item_context)
    validated = validate_classification(text, item_context, proposed)
    classification_rows.append({
        "scenario": name,
        "intent": validated.intent.value,
        "confidence": validated.ambiguity.value,
        "risk": validated.risk_tier.value,
        "side_effect": validated.side_effect_level.value,
        "approval": validated.requires_human_approval,
    })
pd.DataFrame(classification_rows)

## 5. Architecture admission and capability attenuation

Policy chooses among direct function, workflow, bounded agent, pipeline, team, and human escalation. The contract's capabilities must be a subset of the caller's capabilities. Choosing a more elaborate architecture never creates authority.

In [ ]:
decision_rows = []
for name, text in REQUESTS.items():
    item_context = build_context(f"decision-{name}")
    classification = propose_classification(text, item_context)
    decision = decide_architecture(item_context, classification)
    contract = build_execution_contract(item_context, classification, decision)
    decision_rows.append({
        "scenario": name,
        "architecture": decision.architecture.value,
        "reason": decision.reason_codes[0].value,
        "capabilities": len(contract.allowed_capabilities),
        "model_budget": contract.max_model_calls,
        "tool_budget": contract.max_tool_calls,
        "mode": contract.execution_mode.value,
    })
pd.DataFrame(decision_rows)

## 6. Direct function: exact work, zero model calls

“Is checkout healthy?” is one tenant-scoped lookup. Direct does not mean ungoverned: capability, tenant, dependency, output, and audit controls still apply.

In [ ]:
direct_run = run_control_plane(REQUESTS["status"], build_context("direct-demo"))
direct_run.result.model_dump(mode="json")

## 7. Durable password-reset workflow

Workflows govern known state transitions; they are not restricted to linear steps and are not automatically infallible. Production workflows can branch, wait, retry, run parallel nodes, and compensate.

The fixture persists attempt IDs, attempt count, OTP expiry, logical operation identity, and completion. A restart must not reset security state.

In [ ]:
password_context = build_context("password-demo")
password_state = begin_password_reset(password_context, max_attempts=3)
first_status = submit_otp(password_state, "000000", attempt_id="attempt-001")
{
    "first_status": first_status.value,
    "attempts": password_state.attempt_count,
    "expires_at": password_state.otp_expires_at.isoformat(),
    "logical_operation_id": password_state.logical_operation_id,
}

In [ ]:
restart_path = Path("/tmp/course04-password-state.json")
save_password_state(password_state, restart_path)
restored = load_password_state(restart_path)
submit_otp(restored, FIXTURE_OTP, attempt_id="attempt-002")
authorize_password_update(restored)
first_write = execute_password_update(restored)
duplicate_write = execute_password_update(restored)
{
    "state": restored.status.value,
    "attempts_survived_restart": restored.attempt_count,
    "first_write_executed": first_write,
    "duplicate_write_executed": duplicate_write,
    "write_count": restored.password_update_count,
}

## 8. Approval-gated rollback

A plan may validly contain an approval-gated action. That does **not** authorize execution. Planning, authorization, and execution are independent control boundaries.

In [ ]:
blocked_rollback = run_control_plane(REQUESTS["rollback"], build_context("rollback-blocked"))
approved_rollback = run_control_plane(
    REQUESTS["rollback"],
    build_context("rollback-approved"),
    validated_approval=True,
)
pd.DataFrame([
    {
        "case": "no validated approval",
        "plan_valid": blocked_rollback.contract.approval_required,
        "status": blocked_rollback.result.status.value,
        "events": blocked_rollback.result.policy_events,
    },
    {
        "case": "validated approval",
        "plan_valid": approved_rollback.contract.approval_required,
        "status": approved_rollback.result.status.value,
        "events": approved_rollback.result.policy_events,
    },
])

## 9. Bounded single agent

The incident agent gets read tools, evidence requirements, and strict call/cost/deadline/replan budgets. It may recommend a controlled rollback proposal; it never receives the rollback capability.

In [ ]:
agent_run = run_control_plane(REQUESTS["diagnosis"], build_context("agent-demo"))
{
    "architecture": agent_run.contract.architecture.value,
    "capabilities": [item.value for item in agent_run.contract.allowed_capabilities],
    "budgets": {
        "model_calls": agent_run.contract.max_model_calls,
        "tool_calls": agent_run.contract.max_tool_calls,
        "cost_usd": agent_run.contract.max_cost_usd,
        "deadline_ms": agent_run.contract.deadline_ms,
    },
    "result": agent_run.result.model_dump(mode="json"),
}

## 10. Pipeline vs team

Proposal → independent review → deterministic gate is a pipeline. Multi-domain evidence collection can justify parallel specialists and synthesis. Typed artifacts converge into review or synthesis; open-ended debate is not the default.

In [ ]:
pipeline_run = run_control_plane(REQUESTS["pipeline"], build_context("pipeline-demo"))
team_run = run_control_plane(REQUESTS["team"], build_context("team-demo"))
pd.DataFrame([
    {
        "pattern": "pipeline",
        "architecture": pipeline_run.result.architecture.value,
        "model_calls": pipeline_run.result.model_calls,
        "total_work_ms": pipeline_run.result.total_work_ms,
        "wall_clock_ms": pipeline_run.result.wall_clock_ms,
    },
    {
        "pattern": "parallel team",
        "architecture": team_run.result.architecture.value,
        "model_calls": team_run.result.model_calls,
        "total_work_ms": team_run.result.total_work_ms,
        "wall_clock_ms": team_run.result.wall_clock_ms,
    },
])

## 11. Governed architecture transitions

A bounded agent may emit an `ARCHITECTURE_ESCALATION_REQUEST`; it cannot upgrade itself. The control plane validates the transition graph, request identity, remaining depth/transition budgets, revised classification, data policy, and authority before issuing a new contract.

In [ ]:
transition, team_decision = build_agent_to_team_escalation(build_context("transition-demo"))
{
    "current": transition.current.value,
    "history": [item.value for item in transition.history],
    "transitions_used": transition.transitions,
    "new_reason": team_decision.reason_codes[0].value,
}

## 12. Outages, data restrictions, and cancellation

Direct/workflow paths remain usable during model outage. Model-driven routes safely escalate. Classifier/router failure fails closed. Restricted team work goes to human review. Cancellation is checked before the next runner, so call counters stay at zero.

In [ ]:
resilience_rows = [
    ("direct / model down", run_control_plane(REQUESTS["status"], build_context("outage-1"), model_available=False)),
    ("agent / model down", run_control_plane(REQUESTS["diagnosis"], build_context("outage-2"), model_available=False)),
    ("classifier down", run_control_plane(REQUESTS["diagnosis"], build_context("outage-3"), classifier_available=False)),
    ("cancel before team", run_control_plane(REQUESTS["team"], build_context("outage-4"), cancelled=True)),
    ("restricted team", run_control_plane(REQUESTS["team"], build_context("outage-5", data_classification=DataClassification.RESTRICTED))),
]
pd.DataFrame([{
    "case": name,
    "architecture": run.decision.architecture.value,
    "status": run.result.status.value,
    "failure": run.result.failure_code.value if run.result.failure_code else None,
    "model_calls": run.result.model_calls,
    "tool_calls": run.result.tool_calls,
} for name, run in resilience_rows])

## 13. Layered output policy

The common gateway validates request/tenant/architecture identity, actual usage, deadline, evidence, output size, and PII action. The regex detector is illustrative; production DLP needs structured detectors and egress controls. Output size is a resource limit, not exfiltration proof.

In [ ]:
sample = "Contact incident-owner@example.com; card 4111 1111 1111 1111"
pd.DataFrame([
    {"action": action.value, "result": apply_pii_policy(sample, action)}
    for action in (PIIAction.ALLOW, PIIAction.MASK, PIIAction.REDACT)
])

## 14. Labelled routing evaluation

Safety evaluation distinguishes architecture validity from optimality. High-risk false negatives receive greater weight. Architecture regret captures avoidable cost, latency, and complexity among compliant choices.

In [ ]:
baseline_metrics = evaluate_routing()
unsafe_metrics = evaluate_routing(
    forced_architectures={"rollback": ArchitectureType.BOUNDED_SINGLE_AGENT},
    forced_risks={"rollback": RiskTier.LOW},
)
pd.DataFrame([
    {"router": "baseline", **baseline_metrics.model_dump()},
    {"router": "unsafe candidate", **unsafe_metrics.model_dump()},
])

## 15. Same-workload economics and regression gate

These numbers are deterministic fixture metadata, not claims about live-model intelligence. Measure production providers and workloads. Track actual cost per successful compliant request and reject safety or success regressions before considering efficiency.

In [ ]:
pd.DataFrame([row.model_dump(mode="json") for row in same_workload_benchmark()])

In [ ]:
baseline = RouterCandidateMetrics(
    success_rate=0.96,
    high_risk_violation_rate=0,
    mean_cost_usd=0.02,
    mean_latency_ms=220,
    complexity_score=5,
)
candidate = baseline.model_copy(update={"mean_latency_ms": 170, "complexity_score": 7})
unsafe = candidate.model_copy(update={"high_risk_violation_rate": 0.01})
{
    "efficient_candidate": architecture_regression_gate(baseline, candidate),
    "unsafe_candidate": architecture_regression_gate(baseline, unsafe),
    "canary_status": low_risk_canary_eligible(propose_classification(REQUESTS["status"], build_context("canary"))),
    "canary_rollback": low_risk_canary_eligible(propose_classification(REQUESTS["rollback"], build_context("canary-rollback"))),
}

## 16. Optional frameworks and final audit

LangGraph and the OpenAI Agents SDK are optional execution adapters. The stable lesson is the framework-neutral contract. The application still owns classification validation, eligibility, authority, budgets, approvals, artifacts, transitions, termination, and completion.

In [ ]:
adapter_status()

In [ ]:
runs = tuple(
    run_control_plane(
        text,
        build_context(f"audit-{name}"),
        validated_approval=(name == "rollback"),
    )
    for name, text in REQUESTS.items()
)
pd.DataFrame(audit_table(runs))

## Final checkpoint

1. A classifier proposes features; it never grants a capability.
2. A valid approval-gated plan is not execution authorization.
3. Workflows govern transitions and failures; they are not only linear and never magically infallible.
4. An agent may request an architecture change but cannot apply it.
5. Total work and wall-clock time are distinct under parallelism.
6. A valid route need not be the optimal compliant route.
7. Router or classifier failure must fail closed.
8. Teams are justified by measured parallelism or isolation benefits.
9. Frameworks orchestrate; the application owns policy and completion.
10. Ship routing changes through offline evaluation, shadow routing, and low-risk canaries.